# Member A — Dunnhumby Data Transformation

Transforms the 8 **Dunnhumby: The Complete Journey** CSVs (Segments 6/7/8: Replenishment-Ready, Cross-Sell Opportunity, Seasonal Purchase) into canonical-schema CSVs (`backend/schema.sql`).

**Spec:** `DUNNHUMBY.md` — mappings with citations, gaps, and open questions.
**Logic:** `common.py` (id namespace + per-dataset date strategy), `dunnhumby.py` (mappings/cleaning), `validate.py` (71 constraint checks).

This notebook orchestrates those modules — the transformation itself is not duplicated here, so there is one source of truth.

Raw files: `../raw/dunnhumby/` · Output: `../processed/dunnhumby/`

## Running the complete transformation

Install dependencies once with `%pip install -r requirements.txt` from this directory.
Run all cells in order. Sources are downloaded in full with KaggleHub's
`dataset_download` method (https://github.com/Kaggle/kagglehub).
Kaggle may require authentication/consent for a dataset; configure your own Kaggle
credentials outside this notebook. Downloads are cached locally by KaggleHub.
No fixture inputs or row samples are used in this notebook.
ML training remains separate and uses the raw per-segment datasets.


In [1]:
import sys
import time
from pathlib import Path

from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data_prep' / 'transform' / 'common.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the project folder or data_prep/transform.')
TRANSFORM_DIR = PROJECT_ROOT / 'data_prep' / 'transform'
sys.path.insert(0, str(TRANSFORM_DIR))


from common import uid, DEMO_STORE_ID
from dunnhumby import ANCHOR, COLUMNS, RAW_FILES, load_raw, transform, stream_causal
from validate import validate

def raw_dataset_path(handle, required_files):
    """Use local raw files when present; otherwise download from Kaggle."""
    dataset_dir = PROJECT_ROOT / 'data_prep' / 'raw' / handle.split('/')[-1]
    if all(any((dataset_dir / name).is_file() for name in alternatives)
           for alternatives in required_files):
        print(f'Using local dataset: {dataset_dir}')
        return dataset_dir
    import kagglehub
    print(f'Downloading dataset to: {dataset_dir}')
    return Path(kagglehub.dataset_download(handle, output_dir=str(dataset_dir)))

RAW_DIR = raw_dataset_path('frtgnn/dunnhumby-the-complete-journey', [(name,) for name in RAW_FILES])
OUT_DIR = PROJECT_ROOT / 'data_prep' / 'processed' / 'dunnhumby-the-complete-journey'
LOAD_ORDER = ['stores', 'customers', 'products', 'orders', 'order_items',
              'coupon_campaigns', 'coupon_redemptions', 'causal_data']
STORE_ID = DEMO_STORE_ID

print(f'anchor  : {ANCHOR}')
print(f'source  : {RAW_DIR}')
print(f'output  : {OUT_DIR}')
print(f'sources : {", ".join(RAW_FILES)}')

Using local dataset: d:\CDP -FYP\data_prep\raw\dunnhumby-the-complete-journey
anchor  : 2016-01-01
source  : d:\CDP -FYP\data_prep\raw\dunnhumby-the-complete-journey
output  : d:\CDP -FYP\data_prep\processed\dunnhumby-the-complete-journey
sources : transaction_data.csv, hh_demographic.csv, product.csv, causal_data.csv, coupon.csv, coupon_redempt.csv, campaign_table.csv, campaign_desc.csv


In [2]:
raw = load_raw(RAW_DIR, stream_causal=True)
# The large promotion file is read in chunks in the transformation cell.
for name, df in raw.items():
    print(f'{name:<24} {len(df):>12,} rows')

transaction_data.csv        2,595,732 rows
hh_demographic.csv                801 rows
product.csv                    92,353 rows
causal_data.csv                     0 rows
coupon.csv                    124,548 rows
coupon_redempt.csv              2,318 rows
campaign_table.csv              7,208 rows
campaign_desc.csv                  30 rows


In [3]:
started = time.time()
from contract import stage_directory
a_stage = stage_directory(OUT_DIR)
result = transform(raw, STORE_ID)
result.frames.pop('causal_data')  # Replaced by the full streamed CSV.
causal_path = a_stage / 'causal_data' / 'causal_data.csv'
causal_count = stream_causal(RAW_DIR, result.frames['products'], STORE_ID, causal_path, result)
print(f'transformed in {time.time() - started:.1f}s\n')
for table in LOAD_ORDER:
    count = causal_count if table == 'causal_data' else len(result.frames[table])
    print(f'  {table:<20} {count:>10,} rows')

transformed in 1486.6s

  stores                        1 rows
  customers                 2,500 rows
  products                 92,353 rows
  orders                  276,484 rows
  order_items           2,595,732 rows
  coupon_campaigns             30 rows
  coupon_redemptions        2,318 rows
  causal_data          36,786,524 rows


In [4]:
report = validate(result.frames, result.raw_counts, result.dropped,
                  result.dropped_by_source, anchor=ANCHOR,
                  columns={t: c for t, c in COLUMNS.items() if t != 'causal_data'})
print(report.render())

VALIDATION: PASS
  61 checks passed, 4 warnings, 0 failures
  [WARN] row accounting: transaction_data.csv 2,595,732 raw lines -> 2,595,732 order_items, 0 line-level drops recorded
  [WARN] orders.region is entirely NULL — Open Decision #4 (no geo column in this dataset); Revenue by Region / Orders by Region KPI cards will be empty for this tenant
  [WARN] products.unit_price is entirely NULL — Open Decision #11; Average Product Price KPI will be empty for this tenant
  [WARN] drops by source file -> transaction_data.csv: 0, causal_data.csv: 0, coupon_redempt.csv: 0


In [5]:
from contract import publish
manifest = publish(
    OUT_DIR, result.frames, [RAW_DIR / name for name in RAW_FILES], result.raw_counts,
    dropped=result.dropped, notes=result.notes, source_exceptions=result.exceptions,
    stage=a_stage, streams={'causal_data': causal_path},
    expected_counts={'order_items': len(raw['transaction_data.csv']),
                     'coupon_redemptions': len(raw['coupon_redempt.csv']),
                     'causal_data': result.raw_counts['causal_data.csv']},
    sidecars={'coupon_product_mapping': raw['coupon.csv'],
              'campaign_exposure': raw['campaign_table.csv'],
              'redemption_source': raw['coupon_redempt.csv'].assign(source_row=raw['coupon_redempt.csv'].index)},
)
print(f'Validated publication: {OUT_DIR}')


OperationalError: database or disk is full